# Data Exploration and Preparation

## Section 3: Exploring and Preparing the Home Credit Dataset

This notebook demonstrates how to:
1. Load and inspect the Home Credit dataset
2. Select the most predictive features
3. Handle data quality issues (DAYS_EMPLOYED anomaly)
4. Convert days to years for interpretability
5. Encode categorical variables
6. Handle missing values
7. Engineer new features (ratios)
8. Prepare clean dataset for model training

**Goal:** Transform 122 raw features into 18 interpretable, model-ready features

## 3.1 — About the Dataset

The Home Credit Default Risk dataset contains 307,511 loan applications with 122 features.

- **Target:** 0 = Repaid, 1 = Default
- **Class imbalance:** ~92% repaid, ~8% default (realistic for credit)
- **Data quality:** Real-world messy (missing values, anomalies, encodings)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## 3.2 — Load and Inspect Data

In [5]:
# Load the raw dataset
df = pd.read_csv('../data/application_train.csv')

print(f"Dataset Shape: {df.shape}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Dataset Shape: (307511, 122)
Rows: 307,511
Columns: 122


In [6]:
# Check target distribution
print("\nTarget Distribution:")
print(df['TARGET'].value_counts(normalize=True))
print(f"\nClass imbalance: {df['TARGET'].mean():.2%} default rate (typical for credit data)")


Target Distribution:
TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64

Class imbalance: 8.07% default rate (typical for credit data)


In [7]:
# Preview first few rows
print("\nFirst 5 rows:")
df.head()


First 5 rows:


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,APARTMENTS_AVG,BASEMENTAREA_AVG,YEARS_BEGINEXPLUATATION_AVG,YEARS_BUILD_AVG,COMMONAREA_AVG,ELEVATORS_AVG,ENTRANCES_AVG,FLOORSMAX_AVG,FLOORSMIN_AVG,LANDAREA_AVG,LIVINGAPARTMENTS_AVG,LIVINGAREA_AVG,NONLIVINGAPARTMENTS_AVG,NONLIVINGAREA_AVG,APARTMENTS_MODE,BASEMENTAREA_MODE,YEARS_BEGINEXPLUATATION_MODE,YEARS_BUILD_MODE,COMMONAREA_MODE,ELEVATORS_MODE,ENTRANCES_MODE,FLOORSMAX_MODE,FLOORSMIN_MODE,LANDAREA_MODE,LIVINGAPARTMENTS_MODE,LIVINGAREA_MODE,NONLIVINGAPARTMENTS_MODE,NONLIVINGAREA_MODE,APARTMENTS_MEDI,BASEMENTAREA_MEDI,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BUILD_MEDI,COMMONAREA_MEDI,ELEVATORS_MEDI,ENTRANCES_MEDI,FLOORSMAX_MEDI,FLOORSMIN_MEDI,LANDAREA_MEDI,LIVINGAPARTMENTS_MEDI,LIVINGAREA_MEDI,NONLIVINGAPARTMENTS_MEDI,NONLIVINGAREA_MEDI,FONDKAPREMONT_MODE,HOUSETYPE_MODE,TOTALAREA_MODE,WALLSMATERIAL_MODE,EMERGENCYSTATE_MODE,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,351000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.018801,-9461,-637,-3648.0,-2120,NaN,1,1,0,1,1,0,Laborers,1.0,2,2,WEDNESDAY,10,0,0,0,0,0,0,Business Entity Type 3,0.083037,0.262949,0.139376,0.0247,0.0369,0.9722,0.6192,0.0143,0.00,0.0690,0.0833,0.1250,0.0369,0.0202,0.0190,0.0000,0.0000,0.0252,0.0383,0.9722,0.6341,0.0144,0.0000,0.0690,0.0833,0.1250,0.0377,0.022,0.0198,0.0,0.0,0.0250,0.0369,0.9722,0.6243,0.0144,0.00,0.0690,0.0833,0.1250,0.0375,0.0205,0.0193,0.0000,0.00,reg oper account,block of flats,0.0149,"Stone, brick",No,2.0,2.0,2.0,2.0,-1134.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,1129500.0,Family,State servant,Higher education,Married,House / apartment,0.003541,-16765,-1188,-1186.0,-291,NaN,1,1,0,1,1,0,Core staff,2.0,1,1,MONDAY,11,0,0,0,0,0,0,School,0.311267,0.622246,NaN,0.0959,0.0529,0.9851,0.7960,0.0605,0.08,0.0345,0.2917,0.3333,0.0130,0.0773,0.0549,0.0039,0.0098,0.0924,0.0538,0.9851,0.8040,0.0497,0.0806,0.0345,0.2917,0.3333,0.0128,0.079,0.0554,0.0,0.0,0.0968,0.0529,0.9851,0.7987,0.0608,0.08,0.0345,0.2917,0.3333,0.0132,0.0787,0.0558,0.0039,0.01,reg oper account,block of flats,0.0714,Block,No,1.0,0.0,1.0,0.0,-828.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,135000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.010032,-19046,-225,-4260.0,-2531,26.0,1,1,1,1,1,0,Laborers,1.0,2,2,MONDAY,9,0,0,0,0,0,0,Government,NaN,0.555912,0.729567,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,

## 3.3 — Select Key Features

We select 15 features based on predictive power and interpretability.
Working with 15 fields is manageable; 122 is not.

In [8]:
selected_features = [
    'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',  # External credit scores
    'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_ID_PUBLISH',  # Time-based features
    'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE',  # Financial
    'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY',  # Demographics
    'CNT_CHILDREN', 'NAME_EDUCATION_TYPE',  # Personal info
]

X = df[selected_features].copy()
y = df['TARGET'].copy()

print(f"Selected features: {len(X.columns)}")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

Selected features: 15
X shape: (307511, 15)
y shape: (307511,)


## 3.5 — Fix DAYS_EMPLOYED Anomaly

The value 365243 (≈1000 years) indicates unemployment.
Replace with NaN (missing) so median imputation handles it properly.

In [9]:
# Check how many rows have the anomalous value
anomalous_count = (X['DAYS_EMPLOYED'] == 365243).sum()
print(f"Anomalous DAYS_EMPLOYED values: {anomalous_count:,} ({anomalous_count/len(X):.1%} of data)")

# Replace anomaly with NaN
X['DAYS_EMPLOYED'] = X['DAYS_EMPLOYED'].replace(365243, np.nan)

print(f"✓ Replaced with NaN for median imputation")

Anomalous DAYS_EMPLOYED values: 55,374 (18.0% of data)
✓ Replaced with NaN for median imputation


## 3.6 — Convert Days to Years

Transform negative days into positive, interpretable years.
- Readability: -14,585 days → 39.9 years
- API consistency: Users provide age in years, not negative days

In [10]:
# DAYS_BIRTH: negative days before application
X['AGE_YEARS'] = (-X['DAYS_BIRTH'] / 365.25).round(1)

# DAYS_EMPLOYED: negative days worked (or NaN for unemployed)
X['YEARS_EMPLOYED'] = (-X['DAYS_EMPLOYED'] / 365.25).round(1)

# DAYS_ID_PUBLISH: years since ID was issued
X['YEARS_ID_PUBLISH'] = (-X['DAYS_ID_PUBLISH'] / 365.25).round(1)

# Drop raw DAYS columns
X = X.drop(columns=['DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_ID_PUBLISH'])

print("✓ Converted days to years")
print(f"Age range: {X['AGE_YEARS'].min():.1f} - {X['AGE_YEARS'].max():.1f} years")
print(f"Employment range: {X['YEARS_EMPLOYED'].min():.1f} - {X['YEARS_EMPLOYED'].max():.1f} years")

✓ Converted days to years
Age range: 20.5 - 69.1 years
Employment range: -0.0 - 49.0 years


## 3.7 — Encode Categorical Variables

ML models only understand numbers. Convert categorical to numeric.

In [11]:
# Binary encoding: M/F → 0/1
X['CODE_GENDER'] = X['CODE_GENDER'].map({'M': 0, 'F': 1}).fillna(0).astype(int)

# Binary encoding: N/Y → 0/1
X['FLAG_OWN_CAR'] = X['FLAG_OWN_CAR'].map({'N': 0, 'Y': 1}).astype(int)
X['FLAG_OWN_REALTY'] = X['FLAG_OWN_REALTY'].map({'N': 0, 'Y': 1}).astype(int)

# Ordinal encoding for education (preserves ordering: 0 < 1 < 2 < 3 < 4)
education_map = {
    'Lower secondary': 0,
    'Secondary / secondary special': 1,
    'Incomplete higher': 2,
    'Higher education': 3,
    'Academic degree': 4,
}
X['EDUCATION_LEVEL'] = X['NAME_EDUCATION_TYPE'].map(education_map).fillna(1).astype(int)
X = X.drop(columns=['NAME_EDUCATION_TYPE'])

print("✓ Encoded categorical variables")

✓ Encoded categorical variables


## 3.8 — Handle Missing Values

In [12]:
# Check missing values
print("Missing values before filling:")
missing = X.isnull().sum()
if missing.sum() > 0:
    print(missing[missing > 0])
    print(f"\nTotal missing: {missing.sum()} cells")
else:
    print("No missing values")

Missing values before filling:
EXT_SOURCE_1       173378
EXT_SOURCE_2          660
EXT_SOURCE_3        60965
AMT_ANNUITY            12
AMT_GOODS_PRICE       278
YEARS_EMPLOYED      55374
dtype: int64

Total missing: 290667 cells


In [13]:
# Fill with median (robust to outliers)
X = X.fillna(X.median())

print(f"✓ Filled missing values with median")
print(f"Missing values after filling: {X.isnull().sum().sum()}")

✓ Filled missing values with median
Missing values after filling: 0


## 3.9 — Feature Engineering

Create three ratio features that capture financial burden.

- **CREDIT_INCOME_RATIO:** How many years of income is the loan?
- **ANNUITY_INCOME_RATIO:** What % of income goes to monthly payments?
- **CREDIT_GOODS_RATIO:** What % of purchase is financed?

In [14]:
X['CREDIT_INCOME_RATIO'] = X['AMT_CREDIT'] / (X['AMT_INCOME_TOTAL'] + 1)
X['ANNUITY_INCOME_RATIO'] = X['AMT_ANNUITY'] / (X['AMT_INCOME_TOTAL'] + 1)
X['CREDIT_GOODS_RATIO'] = X['AMT_CREDIT'] / (X['AMT_GOODS_PRICE'] + 1)

print("✓ Engineered 3 ratio features")
print(f"\nFeature statistics:")
print(f"  CREDIT_INCOME_RATIO:    mean={X['CREDIT_INCOME_RATIO'].mean():.2f}, max={X['CREDIT_INCOME_RATIO'].max():.2f}")
print(f"  ANNUITY_INCOME_RATIO:   mean={X['ANNUITY_INCOME_RATIO'].mean():.2f}, max={X['ANNUITY_INCOME_RATIO'].max():.2f}")
print(f"  CREDIT_GOODS_RATIO:     mean={X['CREDIT_GOODS_RATIO'].mean():.2f}, max={X['CREDIT_GOODS_RATIO'].max():.2f}")

✓ Engineered 3 ratio features

Feature statistics:
  CREDIT_INCOME_RATIO:    mean=3.96, max=84.73
  ANNUITY_INCOME_RATIO:   mean=0.18, max=1.88
  CREDIT_GOODS_RATIO:     mean=1.12, max=6.00


## 3.10 — Verify Final Feature Set

In [15]:
print(f"Final dataset shape: {X.shape}")
print(f"\nFeatures ({X.shape[1]} total):")
for i, col in enumerate(X.columns, 1):
    print(f"  {i:2d}. {col}")

print(f"\nData types:")
print(X.dtypes)

print(f"\nMissing values: {X.isnull().sum().sum()}")
print(f"Duplicates: {X.duplicated().sum()}")

Final dataset shape: (307511, 18)

Features (18 total):
   1. EXT_SOURCE_1
   2. EXT_SOURCE_2
   3. EXT_SOURCE_3
   4. AMT_INCOME_TOTAL
   5. AMT_CREDIT
   6. AMT_ANNUITY
   7. AMT_GOODS_PRICE
   8. CODE_GENDER
   9. FLAG_OWN_CAR
  10. FLAG_OWN_REALTY
  11. CNT_CHILDREN
  12. AGE_YEARS
  13. YEARS_EMPLOYED
  14. YEARS_ID_PUBLISH
  15. EDUCATION_LEVEL
  16. CREDIT_INCOME_RATIO
  17. ANNUITY_INCOME_RATIO
  18. CREDIT_GOODS_RATIO

Data types:
EXT_SOURCE_1            float64
EXT_SOURCE_2            float64
EXT_SOURCE_3            float64
AMT_INCOME_TOTAL        float64
AMT_CREDIT              float64
AMT_ANNUITY             float64
AMT_GOODS_PRICE         float64
CODE_GENDER               int64
FLAG_OWN_CAR              int64
FLAG_OWN_REALTY           int64
CNT_CHILDREN              int64
AGE_YEARS               float64
YEARS_EMPLOYED          float64
YEARS_ID_PUBLISH        float64
EDUCATION_LEVEL           int64
CREDIT_INCOME_RATIO     float64
ANNUITY_INCOME_RATIO    float64
CREDIT_GOODS

In [16]:
# Save for model training
# These will be split into train/test in the next notebook
print("\n✓ Data preparation complete!")
print(f"Ready for model training: X={X.shape}, y={y.shape}")


✓ Data preparation complete!
Ready for model training: X=(307511, 18), y=(307511,)
